# Phase 05C: Teacher A → S1 Feature-KD Training Readiness

## Mục tiêu

Khóa production engine dùng chung cho hai arm:

- A2: `CE + λf·Feature`;
- A3: `(1−α)·CE + α·T²·KL + λf·Feature`.

Notebook kiểm tra data pipeline, A2/A3 optimizer steps, Validation/Beam-3,
atomic checkpoint, resume-load exact và discarded pilots. Chỉ khi toàn bộ
gate đạt mới authorize Phase 05D và 05E.

## Tiến độ khi bắt đầu

- Hoàn tất: Phase 1–4, Phase 05A, Phase 05B.
- Hiện tại: Phase 05C readiness.
- Kế tiếp: Phase 05D A2 và Phase 05E A3 seed 42.
- Held-out Test: `SEALED`.

## Phù hợp đề cương

Phase 05C phục vụ trực tiếp các yêu cầu áp dụng KD, cài đặt thuật toán,
đánh giá caption bằng metric chuẩn và phân tích chi phí tính toán. ONNX/INT8,
mobile latency/RAM và dữ liệu thực tế chỉ thực hiện sau khi khóa model thắng.

## Inputs

1. Phase 05A output.
2. Phase 05B output.
3. Phase 01 formal output.
4. Phase 02A shared bundle.
5. Phase 03D Teacher A seed-42 output.
6. Phase 03E Teacher A aggregate.
7. MS COCO 2014 images.
8. Offline CLIP ViT-B/32.
9. Offline `pycocoevalcap==1.2` wheel nếu runtime chưa có package.

GPU T4 hoặc mạnh hơn, Internet Off, chạy Cell 1 → Cell 7.


## Cell 1 — Authorization, integrity, progress và proposal alignment

Xác minh Phase 05A/05B, loss contract, manifest, GPU và Test boundary.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase5_teacher_a_s1_feature_kd_training_readiness"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RESULT_DIR = OUTPUT_ROOT / "results"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, LOG_DIR, RESULT_DIR,
    VALIDATION_DIR, CHECKPOINT_DIR, MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


def print_project_progress(
    cell, total, current, next_step, thesis_alignment
):
    print("-" * 72)
    print("CURRENT PROJECT PROGRESS")
    print("-" * 72)
    print("Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B")
    print("Current                   : Phase 05C -", current)
    print("Notebook progress         :", f"{cell} / {total} code cells")
    print("Next                      :", next_step)
    print("Thesis alignment          :", thesis_alignment)
    print("Held-out Test             : SEALED")


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA không khả dụng. Phase 05C bắt buộc GPU. "
        "Trong Kaggle chọn GPU T4 hoặc mạnh hơn, khởi động lại "
        "session và xác minh torch.cuda.is_available() == True."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE5B_COMPLETE = exactly_one(
    "PHASE5_TEACHER_A_S1_FEATURE_KD_GPU_PREFLIGHT_COMPLETE.json"
)
PHASE5C_AUTHORIZED = exactly_one(
    "PHASE5_TEACHER_A_S1_FEATURE_KD_"
    "TRAINING_READINESS_AUTHORIZED.json"
)
PHASE5A_COMPLETE = exactly_one(
    "PHASE5_KD_IMPROVEMENT_ENTRY_COMPLETE.json"
)
PHASE2_READY = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE1_READY = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
TEACHER_RUN_COMPLETE = exactly_one(
    "TEACHER_A_SEED42_FULL_RUN_COMPLETE.json"
)
TEACHER_AGGREGATE_COMPLETE = exactly_one(
    "PHASE3_TEACHER_A_THREE_SEED_COMPLETE.json"
)

phase5b_complete = read_json(PHASE5B_COMPLETE)
phase5c_authorized = read_json(PHASE5C_AUTHORIZED)
phase5a_complete = read_json(PHASE5A_COMPLETE)
phase2_ready = read_json(PHASE2_READY)
teacher_run = read_json(TEACHER_RUN_COMPLETE)
teacher_aggregate = read_json(TEACHER_AGGREGATE_COMPLETE)

assert phase5b_complete["status"] == "complete"
assert phase5b_complete["arms_verified"] == ["A2", "A3"]
assert phase5b_complete["checkpoint_strict_loads_passed"] == 5
assert phase5b_complete["loss_semantics_passed"] is True
assert phase5b_complete["feature_gradient_passed"] is True
assert phase5b_complete["one_batch_overfit_passed"] is True
assert phase5b_complete["fp16_repeatability_passed"] is True
assert phase5b_complete[
    "phase05c_training_readiness_authorized"
] is True
assert phase5b_complete["full_training_authorized"] is False
assert phase5c_authorized["status"] == "authorized"
assert phase5c_authorized["arms"] == ["A2", "A3"]
assert phase5c_authorized["temperature"] == 4.0
assert phase5c_authorized["alpha"] == 0.5
assert phase5c_authorized["lambda_feature"] == 0.1
assert phase5c_authorized["full_training_authorized"] is False
assert phase5a_complete["status"] == "complete"
assert phase5a_complete["anchor_pair"] == "Teacher A -> S1"

for marker in [
    phase5b_complete, phase5c_authorized, phase5a_complete,
    teacher_run, teacher_aggregate,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

PHASE5B_ROOT = PHASE5B_COMPLETE.parent
authorization_ref = phase5b_complete["authorization_marker"]
assert PHASE5C_AUTHORIZED == resolve_inside(
    PHASE5B_ROOT, authorization_ref["relative_path"]
)
assert sha256_file(PHASE5C_AUTHORIZED) == (
    authorization_ref["sha256"]
)
verified_loss_ref = phase5c_authorized[
    "verified_loss_contract"
]
VERIFIED_LOSS_CONTRACT = resolve_inside(
    PHASE5B_ROOT, verified_loss_ref["relative_path"]
)
assert sha256_file(VERIFIED_LOSS_CONTRACT) == (
    verified_loss_ref["sha256"]
)


def verify_manifest(root, csv_path, sums_path):
    rows = list(csv.DictReader(
        Path(csv_path).open("r", encoding="utf-8", newline="")
    ))
    expected = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        expected.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    actual = [
        line.strip()
        for line in Path(sums_path).read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual == expected
    return rows


phase5b_manifest = resolve_inside(
    PHASE5B_ROOT,
    phase5b_complete["manifest"]["csv"]["relative_path"],
)
phase5b_sums = resolve_inside(
    PHASE5B_ROOT,
    phase5b_complete["manifest"][
        "sha256sums"
    ]["relative_path"],
)
assert sha256_file(phase5b_manifest) == (
    phase5b_complete["manifest"]["csv"]["sha256"]
)
assert sha256_file(phase5b_sums) == (
    phase5b_complete["manifest"]["sha256sums"]["sha256"]
)
phase5b_rows = verify_manifest(
    PHASE5B_ROOT, phase5b_manifest, phase5b_sums
)
assert len(phase5b_rows) == 11

write_json(
    AUDIT_DIR / "phase5c_cell01_authorization.json",
    {
        "audit_version": "phase5c_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "phase5b_manifest_rows": len(phase5b_rows),
        "strict_deterministic": True,
        "arms": ["A2", "A3"],
        "temperature": 4.0,
        "alpha": 0.5,
        "lambda_feature": 0.1,
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 05C CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 05B manifest        : 11 / 11 VERIFIED")
print("Preflight gates           : ALL PASSED")
print("Phase 05C authorization   : VERIFIED")
print("Arms                      : A2 / A3")
print("T / alpha / lambda_f      : 4.0 / 0.5 / 0.1")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Strict deterministic      : True")
print("Full training authorized  : False")
print("Test Set status           : SEALED")
print_project_progress(
    cell=1,
    total=7,
    current="Authorization and integrity verified",
    next_step="Production datasets and loaders",
    thesis_alignment=(
        "Preserves reproducibility and leakage control required "
        "for a defensible KD experiment."
    ),
)


PHASE 05C CELL 1 STATUS
Status                    : PASSED
Phase 05B manifest        : 11 / 11 VERIFIED
Preflight gates           : ALL PASSED
Phase 05C authorization   : VERIFIED
Arms                      : A2 / A3
T / alpha / lambda_f      : 4.0 / 0.5 / 0.1
GPU                       : Tesla T4
Strict deterministic      : True
Full training authorized  : False
Test Set status           : SEALED
------------------------------------------------------------------------
CURRENT PROJECT PROGRESS
------------------------------------------------------------------------
Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B
Current                   : Phase 05C - Authorization and integrity verified
Notebook progress         : 1 / 7 code cells
Next                      : Production datasets and loaders
Thesis alignment          : Preserves reproducibility and leakage control required for a defensible KD experiment.
Held-out Test             : SEALED


## Cell 2 — Production datasets, shared geometry và sampler

Khóa Train+Restval 113.287 ảnh, Validation 5.000 ảnh, image-balanced cyclic
caption sampling và stateless augmentation.


In [2]:
CELL_STARTED_AT = time.perf_counter()
from collections import defaultdict

import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
VALIDATION_PARQUET = exactly_one("mscoco_validation.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
    filters=[
        ("experiment_split", "in", ["train", "validation"])
    ],
)
TRAIN_PATHS = {}
VALIDATION_PATHS = {}
for row in path_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    path = leaf / Path(row["image_path"]).name
    target = (
        TRAIN_PATHS
        if row["experiment_split"] == "train"
        else VALIDATION_PATHS
    )
    target[image_id] = path
assert len(TRAIN_PATHS) == 113287
assert len(VALIDATION_PATHS) == 5000

TRAIN_CAPTIONS = defaultdict(list)
for batch_arrow in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=16384,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in batch_arrow.to_pylist():
        TRAIN_CAPTIONS[int(row["image_id"])].append(
            (
                int(row["annotation_id"]),
                tuple(int(value) for value in row["token_ids"]),
            )
        )
assert len(TRAIN_CAPTIONS) == 113287
for image_id in TRAIN_CAPTIONS:
    TRAIN_CAPTIONS[image_id].sort()

VALIDATION_RECORDS = []
VALIDATION_REFERENCES = defaultdict(list)
for batch_arrow in pq.ParquetFile(VALIDATION_PARQUET).iter_batches(
    batch_size=8192,
    columns=["annotation_id", "image_id", "token_ids", "caption"],
):
    for row in batch_arrow.to_pylist():
        image_id = int(row["image_id"])
        VALIDATION_RECORDS.append(
            (
                int(row["annotation_id"]),
                image_id,
                tuple(int(value) for value in row["token_ids"]),
            )
        )
        VALIDATION_REFERENCES[image_id].append(row["caption"])
assert len(VALIDATION_RECORDS) == 25010
assert len(VALIDATION_REFERENCES) == 5000

TRAIN_IMAGE_IDS = tuple(sorted(TRAIN_PATHS))
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_PATHS))
CLIP_MEAN = torch.tensor(
    (0.48145466, 0.4578275, 0.40821073)
).view(3, 1, 1)
CLIP_STD = torch.tensor(
    (0.26862954, 0.26130258, 0.27577711)
).view(3, 1, 1)
IMAGENET_MEAN = torch.tensor(
    (0.485, 0.456, 0.406)
).view(3, 1, 1)
IMAGENET_STD = torch.tensor(
    (0.229, 0.224, 0.225)
).view(3, 1, 1)


def sample_seed(epoch, image_id):
    text = f"{SEED}|{epoch}|{image_id}|phase5c"
    return int.from_bytes(
        hashlib.sha256(text.encode("utf-8")).digest()[:8],
        "big",
    ) % (2 ** 63 - 1)


def shared_train_geometry(image, epoch, image_id):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sample_seed(epoch, image_id))
        top, left, height, width = (
            transforms.RandomResizedCrop.get_params(
                image, scale=(0.80, 1.00), ratio=(0.75, 4 / 3)
            )
        )
        do_flip = bool(torch.rand(()) < 0.5)
    image = TF.resized_crop(
        image, top, left, height, width,
        size=[224, 224],
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    if do_flip:
        image = TF.hflip(image)
    return TF.to_tensor(image)


def validation_geometry(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    image = TF.center_crop(image, [224, 224])
    return TF.to_tensor(image)


def dual_normalize(base):
    return (
        (base - CLIP_MEAN) / CLIP_STD,
        (base - IMAGENET_MEAN) / IMAGENET_STD,
    )


class ProductionTrainDataset(Dataset):
    def __init__(self):
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(TRAIN_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TRAIN_IMAGE_IDS[index]
        captions = TRAIN_CAPTIONS[image_id]
        offset = image_id % len(captions)
        annotation_id, token_ids = captions[
            (self.epoch + offset) % len(captions)
        ]
        with Image.open(TRAIN_PATHS[image_id]) as image:
            base = shared_train_geometry(
                image.convert("RGB"), self.epoch, image_id
            )
        teacher_image, student_image = dual_normalize(base)
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "teacher_image": teacher_image,
            "student_image": student_image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationCaptionDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_RECORDS)

    def __getitem__(self, index):
        annotation_id, image_id, token_ids = VALIDATION_RECORDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as image:
            base = validation_geometry(image.convert("RGB"))
        _, student_image = dual_normalize(base)
        return {
            "image_id": image_id,
            "annotation_id": annotation_id,
            "student_image": student_image,
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
        }


class ValidationImageDataset(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        with Image.open(VALIDATION_PATHS[image_id]) as image:
            base = validation_geometry(image.convert("RGB"))
        _, student_image = dual_normalize(base)
        return {"image_id": image_id, "student_image": student_image}


class EpochCursorSampler(Sampler):
    def __init__(self, length, epoch=0, cursor=0):
        self.length = int(length)
        self.epoch = int(epoch)
        self.cursor = int(cursor)

    def __iter__(self):
        generator = torch.Generator().manual_seed(
            SEED + self.epoch * 1000003
        )
        order = torch.randperm(
            self.length, generator=generator
        ).tolist()
        return iter(order[self.cursor:])

    def __len__(self):
        return self.length - self.cursor


def caption_collate(rows):
    maximum = max(row["token_ids"].numel() for row in rows)
    full = torch.zeros(len(rows), maximum, dtype=torch.long)
    for index, row in enumerate(rows):
        full[index, :row["token_ids"].numel()] = row["token_ids"]
    result = {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [row["annotation_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
        "decoder_input_ids": full[:, :-1],
        "labels": full[:, 1:],
    }
    result["decoder_attention_mask"] = (
        result["decoder_input_ids"].ne(0)
    )
    if "teacher_image" in rows[0]:
        result["teacher_images"] = torch.stack(
            [row["teacher_image"] for row in rows]
        )
    return result


def image_collate(rows):
    return {
        "image_ids": torch.tensor(
            [row["image_id"] for row in rows], dtype=torch.long
        ),
        "student_images": torch.stack(
            [row["student_image"] for row in rows]
        ),
    }


TRAIN_DATASET = ProductionTrainDataset()
VALIDATION_CAPTION_DATASET = ValidationCaptionDataset()
VALIDATION_IMAGE_DATASET = ValidationImageDataset()


def build_train_loader(epoch, cursor=0):
    TRAIN_DATASET.set_epoch(epoch)
    sampler = EpochCursorSampler(
        len(TRAIN_DATASET), epoch=epoch, cursor=cursor
    )
    return DataLoader(
        TRAIN_DATASET,
        batch_size=32,
        sampler=sampler,
        num_workers=2,
        pin_memory=True,
        persistent_workers=False,
        collate_fn=caption_collate,
    )


VALIDATION_CAPTION_LOADER = DataLoader(
    VALIDATION_CAPTION_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=caption_collate,
)
VALIDATION_IMAGE_LOADER = DataLoader(
    VALIDATION_IMAGE_DATASET,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

loader_a = build_train_loader(0)
loader_b = build_train_loader(0)
batch_a = next(iter(loader_a))
batch_b = next(iter(loader_b))
for key in [
    "image_ids", "annotation_ids", "teacher_images",
    "student_images", "decoder_input_ids", "labels",
]:
    assert torch.equal(batch_a[key], batch_b[key])
assert len(set(batch_a["image_ids"].tolist())) == 32

write_json(
    AUDIT_DIR / "phase5c_cell02_data_pipeline.json",
    {
        "audit_version": "phase5c_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "train_images_per_epoch": len(TRAIN_DATASET),
        "captions_per_image_per_epoch": 1,
        "validation_captions": len(VALIDATION_CAPTION_DATASET),
        "validation_images": len(VALIDATION_IMAGE_DATASET),
        "physical_batch_size": 32,
        "shared_train_geometry": True,
        "stateless_augmentation": True,
        "two_loader_exact": True,
        "resume_cursor_supported": True,
        "validation_caption_text_read": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 05C CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Train images / epoch      :", len(TRAIN_DATASET))
print("Captions / image / epoch  : 1")
print("Validation captions       :", len(VALIDATION_CAPTION_DATASET))
print("Validation images         :", len(VALIDATION_IMAGE_DATASET))
print("Physical/effective batch  : 32 / 64")
print("Shared train geometry     : True")
print("Stateless augmentation    : True")
print("Two-loader exact          : True")
print("Resume cursor supported   : True")
print("Test Set status           : SEALED")
print("Next                      : model and production optimizer")


print_project_progress(
    cell=2,
    total=7,
    current="Production data pipeline verified",
    next_step="Model, A2/A3 optimizer and step engines",
    thesis_alignment=(
        "Directly supports the proposal requirement to implement "
        "the algorithm on a standard image-caption dataset."
    ),
)


PHASE 05C CELL 2 STATUS
Status                    : PASSED
Train images / epoch      : 113287
Captions / image / epoch  : 1
Validation captions       : 25010
Validation images         : 5000
Physical/effective batch  : 32 / 64
Shared train geometry     : True
Stateless augmentation    : True
Two-loader exact          : True
Resume cursor supported   : True
Test Set status           : SEALED
Next                      : model and production optimizer
------------------------------------------------------------------------
CURRENT PROJECT PROGRESS
------------------------------------------------------------------------
Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B
Current                   : Phase 05C - Production data pipeline verified
Notebook progress         : 2 / 7 code cells
Next                      : Model, A2/A3 optimizer and step engines
Thesis alignment          : Directly supports the proposal requirement to implement the algorithm on a standard image-caption dat

## Cell 3 — Models, A2/A3 optimizer và step engines

Strict-load Teacher/Student, khóa AdamW/warmup-cosine/FP16, và định nghĩa
token-normalized losses cho cả A2 lẫn A3.


In [3]:
CELL_STARTED_AT = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large
from transformers import CLIPModel


PHASE2_ROOT = PHASE2_READY.parent
S1_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s1_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
decoder_config = read_json(DECODER_CONTRACT)["architecture"]

registry_path = resolve_inside(
    TEACHER_AGGREGATE_COMPLETE.parent,
    teacher_aggregate["checkpoint_registry"]["relative_path"],
)
registry = read_json(registry_path)
seed42_registry = [
    row for row in registry["checkpoints"]
    if int(row["seed"]) == 42
][0]
TEACHER_CHECKPOINT = exactly_one("teacher_a_seed42_best.pt")
assert sha256_file(TEACHER_CHECKPOINT) == (
    seed42_registry["checkpoint_sha256"]
)

source_files = registry["pretrained_source_contract"]["source_files"]
clip_candidates = []
for config_path in INPUT_ROOT.rglob("config.json"):
    candidate = config_path.parent
    if all((candidate / name).is_file() for name in source_files):
        if all(
            sha256_file(candidate / name) == metadata["sha256"]
            for name, metadata in source_files.items()
        ):
            clip_candidates.append(candidate.resolve())
clip_candidates = sorted(set(clip_candidates))
assert len(clip_candidates) == 1
CLIP_SOURCE_DIR = clip_candidates[0]


class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask=None):
        batch, length = ids.shape
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(batch, -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ), diagonal=1
        )
        hidden = self.transformer_decoder(
            target, memory, tgt_mask=causal,
            tgt_key_padding_mask=(
                None if mask is None else ~mask.bool()
            ),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )


class TeacherAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, patches):
        return self.output_norm(
            self.projection(patches) + self.position_embedding
        )


class StudentEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        features = self.backbone(images)
        tokens = features.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


full_clip = CLIPModel.from_pretrained(
    CLIP_SOURCE_DIR, local_files_only=True
)
TEACHER_ENCODER = full_clip.vision_model
del full_clip
TEACHER_ADAPTER = TeacherAdapter()
TEACHER_DECODER = SharedDecoder(decoder_config)
STUDENT_ENCODER = StudentEncoder()
STUDENT_DECODER = SharedDecoder(decoder_config)

with torch.serialization.safe_globals([TorchVersion]):
    teacher_ckpt = torch.load(
        TEACHER_CHECKPOINT, map_location="cpu", weights_only=True
    )
    student_ckpt = torch.load(
        S1_INITIAL_CHECKPOINT, map_location="cpu", weights_only=True
    )
    decoder_ckpt = torch.load(
        DECODER_INITIAL_CHECKPOINT,
        map_location="cpu", weights_only=True,
    )
teacher_state = TEACHER_ENCODER.state_dict()
teacher_state.update(teacher_ckpt["encoder_candidate_state"])
TEACHER_ENCODER.load_state_dict(teacher_state, strict=True)
TEACHER_ADAPTER.load_state_dict(
    teacher_ckpt["adapter_state"], strict=True
)
TEACHER_DECODER.load_state_dict(
    teacher_ckpt["decoder_state"], strict=True
)
STUDENT_ENCODER.load_state_dict(
    student_ckpt["state_dict"], strict=True
)
STUDENT_DECODER.load_state_dict(
    decoder_ckpt["state_dict"], strict=True
)
del teacher_ckpt, student_ckpt, decoder_ckpt, teacher_state

STUDENT_ENCODER_INITIAL = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
STUDENT_DECODER_INITIAL = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
for module in [
    TEACHER_ENCODER, TEACHER_ADAPTER, TEACHER_DECODER,
]:
    module.to(device).eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)
STUDENT_ENCODER.to(device)
STUDENT_DECODER.to(device)


def teacher_outputs(batch):
    with torch.no_grad():
        output = TEACHER_ENCODER(
            pixel_values=batch["teacher_images"]
        )
        memory = TEACHER_ADAPTER(
            output.last_hidden_state[:, 1:, :]
        )
        assert memory.shape[1:] == (49, 256)
        logits = TEACHER_DECODER(
            batch["decoder_input_ids"], memory,
            batch["decoder_attention_mask"],
        )
        return memory.detach(), logits.detach()


def student_outputs(batch):
    memory = STUDENT_ENCODER(batch["student_images"])
    assert memory.shape[1:] == (49, 256)
    logits = STUDENT_DECODER(
        batch["decoder_input_ids"], memory,
        batch["decoder_attention_mask"],
    )
    return memory, logits


def teacher_logits(batch):
    return teacher_outputs(batch)[1]


def student_logits(batch):
    return student_outputs(batch)[1]

def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    count = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            count += 1
    assert count == 46


TEMPERATURE = 4.0
ALPHA = 0.5
LAMBDA_FEATURE = 0.1
FEATURE_EPSILON = 1e-6
ARMS = ("A2", "A3")
BACKWARD_DIVISOR = 1024.0
ACCUMULATION = 2
STEPS_PER_EPOCH = math.ceil(113287 / 64)
MAX_EPOCHS = 20
TOTAL_STEPS = STEPS_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = round(TOTAL_STEPS * 0.05)

backbone_parameters = list(STUDENT_ENCODER.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())


def build_optimizer_state():
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-4},
            {"params": adapter_parameters, "lr": 3e-4},
            {"params": decoder_parameters, "lr": 3e-4},
        ],
        betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4,
    )

    def factor(step):
        if step < WARMUP_STEPS:
            return (step + 1) / WARMUP_STEPS
        progress = (
            (step - WARMUP_STEPS)
            / max(1, TOTAL_STEPS - WARMUP_STEPS)
        )
        return 0.05 + 0.95 * 0.5 * (
            1.0 + math.cos(math.pi * min(progress, 1.0))
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=factor
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer, scheduler, scaler


def normalized_feature_loss(student_memory, teacher_memory):
    student_norm = F.normalize(
        student_memory.float(),
        p=2,
        dim=-1,
        eps=FEATURE_EPSILON,
    )
    teacher_norm = F.normalize(
        teacher_memory.detach().float(),
        p=2,
        dim=-1,
        eps=FEATURE_EPSILON,
    )
    return F.mse_loss(
        student_norm, teacher_norm, reduction="mean"
    )


def loss_sums(
    arm,
    student_logits_value,
    teacher_logits_value,
    student_memory,
    teacher_memory,
    labels,
):
    assert arm in ARMS
    valid = labels.ne(0)
    valid_count = int(valid.sum())
    ce_sum = F.cross_entropy(
        student_logits_value.float().reshape(-1, 10080),
        labels.reshape(-1),
        ignore_index=0,
        reduction="sum",
    )
    student_log_probs = F.log_softmax(
        student_logits_value.float() / TEMPERATURE, dim=-1
    )
    teacher_probs = F.softmax(
        teacher_logits_value.float() / TEMPERATURE, dim=-1
    )
    kl_sum = (
        F.kl_div(
            student_log_probs, teacher_probs, reduction="none"
        ).sum(dim=-1)
        * valid
    ).sum() * (TEMPERATURE ** 2)
    feature_mean = normalized_feature_loss(
        student_memory, teacher_memory
    )
    feature_sum = feature_mean * valid_count
    if arm == "A2":
        combined_sum = (
            ce_sum + LAMBDA_FEATURE * feature_sum
        )
    else:
        combined_sum = (
            (1.0 - ALPHA) * ce_sum
            + ALPHA * kl_sum
            + LAMBDA_FEATURE * feature_sum
        )
    return (
        combined_sum, ce_sum, kl_sum,
        feature_sum, valid_count,
    )

def to_device(batch):
    return {
        key: (
            value.to(device, non_blocking=True)
            if isinstance(value, torch.Tensor) else value
        )
        for key, value in batch.items()
    }


def optimizer_step(
    arm, microbatches, optimizer, scheduler, scaler,
    retry_limit=10,
):
    assert arm in ARMS
    retries = 0
    while True:
        set_student_train()
        optimizer.zero_grad(set_to_none=True)
        combined_total = 0.0
        ce_total = 0.0
        kd_total = 0.0
        feature_total = 0.0
        valid_total = 0
        for cpu_batch in microbatches:
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                teacher_memory, teacher_logits_value = (
                    teacher_outputs(current)
                )
                student_memory, student_logits_value = (
                    student_outputs(current)
                )
                (
                    combined, ce_sum, kd_sum,
                    feature_sum, valid,
                ) = loss_sums(
                    arm,
                    student_logits_value,
                    teacher_logits_value,
                    student_memory,
                    teacher_memory,
                    current["labels"],
                )
            scaler.scale(
                combined / BACKWARD_DIVISOR
            ).backward()
            combined_total += float(combined)
            ce_total += float(ce_sum)
            kd_total += float(kd_sum)
            feature_total += float(feature_sum)
            valid_total += valid
        scaler.unscale_(optimizer)
        correction = BACKWARD_DIVISOR / valid_total
        for parameter in (
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters())
        ):
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters()),
            1.0,
        )
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        new_scale = scaler.get_scale()
        if new_scale < old_scale:
            retries += 1
            assert retries <= retry_limit
            continue
        assert torch.isfinite(gradient_norm)
        scheduler.step()
        return {
            "arm": arm,
            "combined_loss": combined_total / valid_total,
            "ce": ce_total / valid_total,
            "kd_kl_t2": kd_total / valid_total,
            "feature_loss": feature_total / valid_total,
            "valid_tokens": valid_total,
            "gradient_norm": float(gradient_norm),
            "overflow_retries": retries,
            "scaler": float(new_scale),
        }


optimizer, scheduler, scaler = build_optimizer_state()
assert STEPS_PER_EPOCH == 1771
assert TOTAL_STEPS == 35420
assert WARMUP_STEPS == 1771

write_json(
    CONFIG_DIR / "phase5c_production_optimizer_contract.json",
    {
        "contract_version": "phase5c_optimizer_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "optimizer": "AdamW",
        "learning_rates": {
            "backbone": 1e-4, "adapter": 3e-4, "decoder": 3e-4,
        },
        "weight_decay": 1e-4,
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "maximum_epochs": MAX_EPOCHS,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "minimum_lr_factor": 0.05,
        "mixed_precision": "FP16 plus GradScaler",
        "gradscaler_initial_scale": 1024.0,
        "overflow_retry_limit": 10,
        "backward_divisor": BACKWARD_DIVISOR,
        "gradient_clip": 1.0,
        "teacher_frozen": True,
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "lambda_feature": LAMBDA_FEATURE,
        "feature_loss": "token-wise L2-normalized MSE",
        "feature_projection": "identity",
        "arms": list(ARMS),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase5c_cell03_model_optimizer.json",
    {
        "audit_version": "phase5c_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "strict_loads": 5,
        "teacher_frozen": True,
        "student_parameters": 8990352,
        "optimizer_parameter_groups": 3,
        "arms": list(ARMS),
        "lambda_feature": LAMBDA_FEATURE,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "total_steps": TOTAL_STEPS,
        "scheduler_advanced": False,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 05C CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Strict checkpoint loads   : 5 / 5 PASSED")
print("Teacher frozen            : True")
print("Temperature / alpha       :", TEMPERATURE, "/", ALPHA)
print("Lambda feature            :", LAMBDA_FEATURE)
print("Arms                      : A2 / A3")
print("Student parameters        : 8990352")
print("Optimizer groups          : 3 / 3")
print("Physical/effective batch  : 32 / 64")
print("Steps / epoch             :", STEPS_PER_EPOCH)
print("Total optimizer steps     :", TOTAL_STEPS)
print("Warmup steps              :", WARMUP_STEPS)
print("FP16 / GradScaler         : ENABLED / 1024")
print("Overflow retry limit      : 10")
print("Token-balanced accumulation: READY")
print("Optimizer step performed  : False")
print("Test Set status           : SEALED")
print("Next                      : Validation and Beam-3 engine")


print_project_progress(
    cell=3,
    total=7,
    current="A2/A3 model, optimizer and step engines verified",
    next_step="Validation CE and Beam-3 engine",
    thesis_alignment=(
        "Implements and isolates two KD variants required for "
        "algorithm analysis and ablation in the thesis."
    ),
)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /kaggle/input/models/nguynthnhdan/clip-vit-base-patch32-offline/pytorch/transformers-pytorch/1/openai_clip-vit-base-patch32_3d74acf9
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PHASE 05C CELL 3 STATUS
Status                    : PASSED
Strict checkpoint loads   : 5 / 5 PASSED
Teacher frozen            : True
Temperature / alpha       : 4.0 / 0.5
Lambda feature            : 0.1
Arms                      : A2 / A3
Student parameters        : 8990352
Optimizer groups          : 3 / 3
Physical/effective batch  : 32 / 64
Steps / epoch             : 1771
Total optimizer steps     : 35420
Warmup steps              : 1771
FP16 / GradScaler         : ENABLED / 1024
Overflow retry limit      : 10
Token-balanced accumulation: READY
Optimizer step performed  : False
Test Set status           : SEALED
Next                      : Validation and Beam-3 engine
------------------------------------------------------------------------
CURRENT PROJECT PROGRESS
------------------------------------------------------------------------
Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B
Current                   : Phase 05C - A2/A3 model, optimizer and step engines verified


## Cell 4 — Validation CE và Beam-3 engine

Xác minh 25.010 Validation captions, 5.000 unique images, official COCO
metrics và deterministic Beam-3. Không chạy full Validation trong readiness.


In [4]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token
            for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value
            for key, value in raw_id_to_token.items()
        }
else:
    token_to_id = vocabulary_payload["token_to_id"]
    ID_TO_TOKEN = {int(value): key for key, value in token_to_id.items()}


def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(
        tokenized_gt, tokenized_pred
    )
    meteor, _ = Meteor().compute_score(
        tokenized_gt, tokenized_pred
    )
    rouge, _ = Rouge().compute_score(
        tokenized_gt, tokenized_pred
    )
    cider, _ = Cider().compute_score(
        tokenized_gt, tokenized_pred
    )
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


smoke_ids = list(VALIDATION_IMAGE_IDS[:32])
smoke_references = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in smoke_ids
}
smoke_hypotheses = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in smoke_ids
}
metric_smoke = compute_coco_metrics(
    smoke_references, smoke_hypotheses
)
assert all(math.isfinite(value) for value in metric_smoke.values())


def teacher_forced_ce(loader, maximum_batches=None):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch_index, cpu_batch in enumerate(loader):
            current = to_device(cpu_batch)
            with torch.amp.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = student_logits(current)
            valid = current["labels"].ne(0)
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                current["labels"].reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
            total_loss += float(loss_sum)
            total_tokens += int(valid.sum())
            if (
                maximum_batches is not None
                and batch_index + 1 >= maximum_batches
            ):
                break
    return total_loss / total_tokens, total_tokens


def beam_generate_one(student_image, beam_size=3):
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        memory = STUDENT_ENCODER(
            student_image.unsqueeze(0).to(device)
        )
        beams = [([1], 0.0, False)]
        for step in range(31):
            candidates = []
            active = [item for item in beams if not item[2]]
            if not active:
                break
            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue
                ids = torch.tensor(
                    tokens, dtype=torch.long, device=device
                ).unsqueeze(0)
                mask = ids.ne(0)
                logits = STUDENT_DECODER(ids, memory, mask)[:, -1]
                log_probs = F.log_softmax(logits.float(), dim=-1)[0]
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, beam_size)
                for value, index in zip(
                    values.tolist(), indices.tolist()
                ):
                    next_tokens = tokens + [index]
                    candidates.append(
                        (
                            next_tokens,
                            score + value,
                            index == 2,
                        )
                    )
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(
                candidates, key=rank, reverse=True
            )[:beam_size]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
        return best[0], best[1]


validation_smoke_batch = next(iter(VALIDATION_IMAGE_LOADER))
tokens_a, score_a = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
tokens_b, score_b = beam_generate_one(
    validation_smoke_batch["student_images"][0]
)
assert tokens_a == tokens_b
assert score_a == score_b
smoke_ce, smoke_tokens = teacher_forced_ce(
    VALIDATION_CAPTION_LOADER, maximum_batches=1
)
assert math.isfinite(smoke_ce)

REFERENCES_PATH = (
    VALIDATION_DIR / "karpathy_validation_references.json"
)
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    AUDIT_DIR / "phase5c_cell04_validation_engine.json",
    {
        "audit_version": "phase5c_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metric_backend": "pycocoevalcap 1.2",
        "metric_smoke": metric_smoke,
        "teacher_forced_smoke_ce": smoke_ce,
        "teacher_forced_smoke_tokens": smoke_tokens,
        "beam_size": 3,
        "length_penalty": 0.7,
        "beam_repeat_exact": True,
        "full_validation_run": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 05C CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("COCO metrics              : 5 / 5 WIRED")
print("Teacher-forced CE smoke   :", smoke_ce)
print("Beam size / penalty       : 3 / 0.7")
print("Beam repeat exact         : True")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Next                      : atomic checkpoint and production pilot")


print_project_progress(
    cell=4,
    total=7,
    current="Validation CE and Beam-3 engine verified",
    next_step="Atomic checkpoints and discarded A2/A3 pilots",
    thesis_alignment=(
        "Implements the proposal requirement to evaluate caption "
        "quality with standard metrics without using Held-out Test."
    ),
)


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 1986 tokens at 19210.52 tokens per second.
PTBTokenizer tokenized 385 tokens at 6036.26 tokens per second.


{'testlen': 328, 'reflen': 328, 'guess': [328, 296, 264, 232], 'correct': [328, 296, 264, 232]}
ratio: 0.9999999999969512
PHASE 05C CELL 4 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
COCO metrics              : 5 / 5 WIRED
Teacher-forced CE smoke   : 11.420501373626374
Beam size / penalty       : 3 / 0.7
Beam repeat exact         : True
Full Validation run       : False
Test Set status           : SEALED
Next                      : atomic checkpoint and production pilot
------------------------------------------------------------------------
CURRENT PROJECT PROGRESS
------------------------------------------------------------------------
Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B
Current                   : Phase 05C - Validation CE and Beam-3 engine verified
Notebook progress         : 4 / 7 code cells
Next                      : Atomic checkpoints and discarded A2/A3 pilots
Thesis alignment          : I

## Cell 5 — Atomic checkpoints và discarded A2/A3 pilots

Chạy 4 optimizer steps cho mỗi arm, kiểm tra save/load chính xác, xóa toàn bộ
checkpoint tạm và phục hồi Student initialization.


In [5]:
CELL_STARTED_AT = time.perf_counter()


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if isinstance(left, torch.Tensor):
        return (
            isinstance(right, torch.Tensor)
            and left.dtype == right.dtype
            and left.shape == right.shape
            and torch.equal(left.cpu(), right.cpu())
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and left.keys() == right.keys()
            and all(
                nested_exact(left[key], right[key])
                for key in left
            )
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(
                nested_exact(a, b)
                for a, b in zip(left, right)
            )
        )
    return left == right


def restore_student_initial():
    STUDENT_ENCODER.load_state_dict(
        STUDENT_ENCODER_INITIAL, strict=True
    )
    STUDENT_DECODER.load_state_dict(
        STUDENT_DECODER_INITIAL, strict=True
    )
    for module in [STUDENT_ENCODER, STUDENT_DECODER]:
        module.zero_grad(set_to_none=True)


def atomic_save(path, payload):
    temporary = Path(str(path) + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_training_checkpoint(
    path, arm, optimizer, scheduler, scaler, progress
):
    atomic_save(
        path,
        {
            "schema_version": (
                "phase5_teacher_a_s1_feature_kd_checkpoint_v1"
            ),
            "experiment_id": f"P5A_A_S1_{arm}",
            "arm": arm,
            "student_encoder_state": clone_cpu(
                STUDENT_ENCODER.state_dict()
            ),
            "student_decoder_state": clone_cpu(
                STUDENT_DECODER.state_dict()
            ),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            "progress": progress,
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "teacher_checkpoint_sha256": sha256_file(
                TEACHER_CHECKPOINT
            ),
            "temperature": TEMPERATURE,
            "alpha": ALPHA,
            "lambda_feature": LAMBDA_FEATURE,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )


all_pilot_rows = []
pilot_summaries = {}
for arm_index, arm in enumerate(ARMS):
    restore_student_initial()
    set_deterministic(SEED + 500 + arm_index)
    optimizer, scheduler, scaler = build_optimizer_state()
    pilot_loader = build_train_loader(epoch=0, cursor=0)
    pilot_iterator = iter(pilot_loader)
    pilot_rows = []
    pilot_started = time.perf_counter()
    cursor = 0
    overflow_total = 0
    for optimizer_step_index in range(1, 5):
        microbatches = [
            next(pilot_iterator) for _ in range(ACCUMULATION)
        ]
        result = optimizer_step(
            arm,
            microbatches,
            optimizer,
            scheduler,
            scaler,
            retry_limit=10,
        )
        cursor += sum(
            len(batch["image_ids"]) for batch in microbatches
        )
        overflow_total += result["overflow_retries"]
        row = {
            "arm": arm,
            "optimizer_step": optimizer_step_index,
            "cursor": cursor,
            **result,
            "lr_backbone": optimizer.param_groups[0]["lr"],
            "lr_adapter": optimizer.param_groups[1]["lr"],
            "lr_decoder": optimizer.param_groups[2]["lr"],
        }
        pilot_rows.append(row)
        all_pilot_rows.append(row)
        print(
            f"arm={arm} | step={optimizer_step_index:02d} | "
            f"cursor={cursor} | "
            f"loss={result['combined_loss']:.6f} | "
            f"ce={result['ce']:.6f} | "
            f"kd={result['kd_kl_t2']:.6f} | "
            f"feature={result['feature_loss']:.6f} | "
            f"grad={result['gradient_norm']:.4f}"
        )
    pilot_seconds = time.perf_counter() - pilot_started
    assert cursor == 256
    assert all(
        math.isfinite(row["combined_loss"])
        for row in pilot_rows
    )
    assert all(
        math.isfinite(row["feature_loss"])
        for row in pilot_rows
    )

    temp_checkpoint = CHECKPOINT_DIR / (
        f"temporary_{arm.lower()}_production_pilot.pt"
    )
    progress = {
        "epoch": 0,
        "cursor": cursor,
        "global_optimizer_step": 4,
    }
    save_training_checkpoint(
        temp_checkpoint,
        arm,
        optimizer,
        scheduler,
        scaler,
        progress,
    )
    expected_encoder = clone_cpu(STUDENT_ENCODER.state_dict())
    expected_decoder = clone_cpu(STUDENT_DECODER.state_dict())
    expected_optimizer = clone_cpu(optimizer.state_dict())
    expected_scheduler = clone_cpu(scheduler.state_dict())
    expected_scaler = clone_cpu(scaler.state_dict())
    expected_cpu_rng = torch.get_rng_state().clone()
    expected_cuda_rng = [
        state.clone() for state in torch.cuda.get_rng_state_all()
    ]

    restore_student_initial()
    loaded_optimizer, loaded_scheduler, loaded_scaler = (
        build_optimizer_state()
    )
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            temp_checkpoint,
            map_location="cpu",
            weights_only=True,
        )
    assert checkpoint["arm"] == arm
    STUDENT_ENCODER.load_state_dict(
        checkpoint["student_encoder_state"], strict=True
    )
    STUDENT_DECODER.load_state_dict(
        checkpoint["student_decoder_state"], strict=True
    )
    loaded_optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )
    loaded_scheduler.load_state_dict(
        checkpoint["scheduler_state"]
    )
    loaded_scaler.load_state_dict(
        checkpoint["scaler_state"]
    )
    torch.set_rng_state(checkpoint["cpu_rng"])
    torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
    assert nested_exact(
        expected_encoder, STUDENT_ENCODER.state_dict()
    )
    assert nested_exact(
        expected_decoder, STUDENT_DECODER.state_dict()
    )
    assert nested_exact(
        expected_optimizer, loaded_optimizer.state_dict()
    )
    assert nested_exact(
        expected_scheduler, loaded_scheduler.state_dict()
    )
    assert nested_exact(
        expected_scaler, loaded_scaler.state_dict()
    )
    assert torch.equal(expected_cpu_rng, torch.get_rng_state())
    assert all(
        torch.equal(left, right)
        for left, right in zip(
            expected_cuda_rng, torch.cuda.get_rng_state_all()
        )
    )
    temp_checkpoint.unlink()
    assert not temp_checkpoint.exists()
    pilot_summaries[arm] = {
        "optimizer_steps": 4,
        "images": cursor,
        "overflow_retries": overflow_total,
        "seconds_per_optimizer_step": pilot_seconds / 4,
        "estimated_minutes_per_epoch": (
            pilot_seconds / 4 * STEPS_PER_EPOCH / 60
        ),
        "resume_load_exact": True,
        "temporary_checkpoint_deleted": True,
    }
    restore_student_initial()

PILOT_CURVE = LOG_DIR / "phase5c_discarded_a2_a3_pilots.csv"
with PILOT_CURVE.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(all_pilot_rows[0])
    )
    writer.writeheader()
    writer.writerows(all_pilot_rows)

write_json(
    AUDIT_DIR / "phase5c_cell05_production_pilots.json",
    {
        "audit_version": "phase5c_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "arms": list(ARMS),
        "summaries": pilot_summaries,
        "total_optimizer_steps": 8,
        "total_images": 512,
        "atomic_checkpoint": True,
        "resume_load_exact": "2 / 2",
        "temporary_checkpoints_deleted": "2 / 2",
        "student_initial_restore_exact": "2 / 2",
        "teacher_frozen": True,
        "pilot_weights_saved": False,
        "full_validation_run": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 05C CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Pilot arms                : A2 / A3")
print("Pilot optimizer steps     : 4 + 4")
print("Pilot images              : 256 + 256")
print("Resume load exact         : 2 / 2")
print("Temporary checkpoints     : DELETED 2 / 2")
print("Initial restore exact     : 2 / 2")
print("Pilot weights saved       : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print_project_progress(
    cell=5,
    total=7,
    current="A2/A3 atomic checkpoint pilots passed",
    next_step="Lock production contracts",
    thesis_alignment=(
        "Verifies feasibility and resource behavior before "
        "expensive full experiments, strengthening reproducibility."
    ),
)


/tmp/ipykernel_23/2050200323.py:408: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  combined_total += float(combined)


arm=A2 | step=01 | cursor=64 | loss=11.246073 | ce=11.245287 | kd=4.652666 | feature=0.007855 | grad=5.0988
arm=A2 | step=02 | cursor=128 | loss=11.251592 | ce=11.250812 | kd=4.660631 | feature=0.007797 | grad=5.1825
arm=A2 | step=03 | cursor=192 | loss=11.174184 | ce=11.173402 | kd=4.647636 | feature=0.007823 | grad=5.2840
arm=A2 | step=04 | cursor=256 | loss=11.181442 | ce=11.180661 | kd=4.764243 | feature=0.007815 | grad=5.1755
arm=A3 | step=01 | cursor=64 | loss=7.948688 | ce=11.242930 | kd=4.652875 | feature=0.007855 | grad=2.6044
arm=A3 | step=02 | cursor=128 | loss=7.958033 | ce=11.253769 | kd=4.660738 | feature=0.007797 | grad=2.6635
arm=A3 | step=03 | cursor=192 | loss=7.900427 | ce=11.152714 | kd=4.646577 | feature=0.007823 | grad=2.6587
arm=A3 | step=04 | cursor=256 | loss=7.980037 | ce=11.194726 | kd=4.763785 | feature=0.007815 | grad=2.6438
PHASE 05C CELL 5 STATUS
Status                    : PASSED
Pilot arms                : A2 / A3
Pilot optimizer steps     : 4 + 4
Pilot

## Cell 6 — Khóa production và ablation contract

Khóa toàn bộ data, optimizer, loss, validation, early stopping, paired
comparisons và phần đối chiếu đề cương.


In [6]:
CELL_STARTED_AT = time.perf_counter()
PRODUCTION_CONTRACT = CONFIG_DIR / (
    "phase5c_teacher_a_s1_feature_kd_production_contract.json"
)
production_contract = {
    "contract_version": (
        "phase5c_teacher_a_s1_feature_kd_production_v1"
    ),
    "created_at_utc": now_utc(),
    "status": "locked",
    "anchor_pair": "Teacher A -> S1",
    "teacher": {
        "id": "A",
        "seed": 42,
        "checkpoint": str(TEACHER_CHECKPOINT),
        "checkpoint_sha256": sha256_file(TEACHER_CHECKPOINT),
        "mode": "eval",
        "frozen": True,
    },
    "student": {
        "id": "S1",
        "seed": 42,
        "initial_encoder_checkpoint": str(S1_INITIAL_CHECKPOINT),
        "initial_decoder_checkpoint": str(
            DECODER_INITIAL_CHECKPOINT
        ),
    },
    "arms": {
        "A2": {
            "loss": "CE + lambda_feature * normalized_feature_MSE",
            "logit_kd": False,
            "feature_kd": True,
        },
        "A3": {
            "loss": (
                "(1-alpha)*CE + alpha*T^2*KL + "
                "lambda_feature*normalized_feature_MSE"
            ),
            "logit_kd": True,
            "feature_kd": True,
        },
    },
    "kd": {
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "lambda_feature": LAMBDA_FEATURE,
        "feature_loss": "token-wise L2-normalized MSE",
        "feature_projection": "identity",
        "teacher_outputs_detached": True,
        "padding_mask": "labels != PAD_ID",
    },
    "training": {
        "images_per_epoch": 113287,
        "caption_sampling": "one per image per epoch cyclic",
        "physical_batch": 32,
        "accumulation": 2,
        "effective_batch": 64,
        "maximum_epochs": 20,
        "optimizer": "AdamW",
        "scheduler": "warmup 5 percent plus cosine",
        "mixed_precision": "FP16 plus GradScaler",
        "gradient_clip": 1.0,
        "overflow_retry_limit": 10,
        "atomic_last_every_optimizer_steps": 250,
        "resume_boundary": "optimizer step",
    },
    "validation": {
        "caption_count": 25010,
        "image_count": 5000,
        "generation": "Beam-3 length penalty 0.7",
        "metrics": [
            "BLEU-1", "BLEU-4", "METEOR",
            "ROUGE-L", "CIDEr-D",
        ],
        "selection_primary": "CIDEr-D",
        "tie_break": ["BLEU-4", "CE", "earlier epoch"],
    },
    "early_stopping": {
        "minimum_epoch": 5,
        "patience": 4,
        "minimum_delta": 0.001,
    },
    "ablation_comparison": {
        "A2_reference": "A0 S1 CE-only seed42",
        "A3_reference": "A1 Teacher A -> S1 Logit-KD seed42",
        "cider_delta_minimum": 0.001,
        "bleu4_non_regression_tolerance": -0.002,
        "teacher_metric_is_not_kd_improvement": True,
    },
    "proposal_alignment": {
        "teacher_student_kd": "directly addressed",
        "caption_quality_evaluation": "directly addressed",
        "resource_measurement": "readiness and full-run logs",
        "mobile_deployment": (
            "deferred until winning Student checkpoint is locked"
        ),
    },
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(PRODUCTION_CONTRACT, production_contract)
write_json(
    AUDIT_DIR / "phase5c_cell06_production_contract.json",
    {
        "audit_version": "phase5c_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "contract": str(PRODUCTION_CONTRACT),
        "contract_sha256": sha256_file(PRODUCTION_CONTRACT),
        "arms": list(ARMS),
        "selection_split": "validation",
        "primary_metric": "CIDEr-D",
        "atomic_last_checkpoint": True,
        "atomic_best_checkpoint": True,
        "mid_epoch_resume": True,
        "proposal_alignment": "verified",
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 05C CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Production contract       : LOCKED")
print("Arms                      : A2 / A3")
print("Maximum epochs            : 20")
print("Selection split           : Validation")
print("Primary metric            : CIDEr-D")
print("Beam size / penalty       : 3 / 0.7")
print("Atomic last / best        : READY / READY")
print("Mid-epoch resume          : READY")
print("Proposal alignment        : VERIFIED")
print("Full training started     : False")
print("Test Set status           : SEALED")
print_project_progress(
    cell=6,
    total=7,
    current="A2/A3 production contract locked",
    next_step="Formal package and full-run authorizations",
    thesis_alignment=(
        "The experiment now maps directly to the thesis KD, "
        "evaluation and resource-analysis requirements."
    ),
)


PHASE 05C CELL 6 STATUS
Status                    : PASSED
Production contract       : LOCKED
Arms                      : A2 / A3
Maximum epochs            : 20
Selection split           : Validation
Primary metric            : CIDEr-D
Beam size / penalty       : 3 / 0.7
Atomic last / best        : READY / READY
Mid-epoch resume          : READY
Proposal alignment        : VERIFIED
Full training started     : False
Test Set status           : SEALED
------------------------------------------------------------------------
CURRENT PROJECT PROGRESS
------------------------------------------------------------------------
Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B
Current                   : Phase 05C - A2/A3 production contract locked
Notebook progress         : 6 / 7 code cells
Next                      : Formal package and full-run authorizations
Thesis alignment          : The experiment now maps directly to the thesis KD, evaluation and resource-analysis requirements.


## Cell 7 — Formal package và full-run authorizations

Tạo manifest và hai authorization độc lập cho Phase 05D A2 và Phase 05E A3.


In [7]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase5c_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase5c_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase5c_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE5_TEACHER_A_S1_FEATURE_KD_"
    "TRAINING_READINESS_COMPLETE.json"
)
A2_AUTHORIZED = OUTPUT_ROOT / (
    "PHASE5_TEACHER_A_S1_A2_SEED42_FULL_RUN_AUTHORIZED.json"
)
A3_AUTHORIZED = OUTPUT_ROOT / (
    "PHASE5_TEACHER_A_S1_A3_SEED42_FULL_RUN_AUTHORIZED.json"
)

for arm, path, next_phase in [
    ("A2", A2_AUTHORIZED, "Phase 05D A2 seed-42 full run"),
    ("A3", A3_AUTHORIZED, "Phase 05E A3 seed-42 full run"),
]:
    write_json(
        path,
        {
            "marker_version": (
                f"phase5_teacher_a_s1_{arm.lower()}_"
                "seed42_full_run_authorized_v1"
            ),
            "created_at_utc": now_utc(),
            "status": "authorized",
            "arm": arm,
            "seed": 42,
            "production_contract": {
                "relative_path": PRODUCTION_CONTRACT.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(PRODUCTION_CONTRACT),
            },
            "next_phase": next_phase,
            "selection_split": "validation",
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON,
            SHA256SUMS, COMPLETE_MARKER,
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase5c_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(
        OUTPUT_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    COMPLETE_MARKER,
    {
        "marker_version": (
            "phase5_teacher_a_s1_feature_kd_"
            "training_readiness_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "anchor_pair": "Teacher A -> S1",
        "arms": ["A2", "A3"],
        "production_data_pipeline": "passed",
        "model_optimizer_engine": "passed",
        "validation_beam_engine": "passed",
        "discarded_pilots": "A2 4/4 and A3 4/4 passed",
        "production_contract": "locked",
        "phase05d_a2_seed42_authorized": True,
        "phase05e_a3_seed42_authorized": True,
        "authorizations": {
            "A2": {
                "relative_path": A2_AUTHORIZED.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(A2_AUTHORIZED),
            },
            "A3": {
                "relative_path": A3_AUTHORIZED.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(A3_AUTHORIZED),
            },
        },
        "production_contract_reference": {
            "relative_path": PRODUCTION_CONTRACT.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(PRODUCTION_CONTRACT),
        },
        "current_progress": {
            "completed": [
                "Phase 1", "Phase 2", "Phase 3", "Phase 4",
                "Phase 05A", "Phase 05B", "Phase 05C",
            ],
            "next": [
                "Phase 05D A2 seed42",
                "Phase 05E A3 seed42",
                "Phase 05F ablation gate",
            ],
        },
        "proposal_alignment": {
            "status": "aligned",
            "covered": [
                "Teacher and Student construction",
                "knowledge distillation",
                "standard caption metrics",
                "resource-aware experimentation",
            ],
            "pending": [
                "winning-model ONNX/INT8 export",
                "mobile latency/RAM benchmark",
                "real-data deployment evaluation",
            ],
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "pilot_optimizer_steps_only": True,
        "pilot_weights_saved": False,
        "full_training_started": False,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 05C CELL 7 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Data pipeline             : PASSED")
print("A2/A3 optimizer engine    : PASSED")
print("Validation/Beam-3 engine  : PASSED")
print("Discarded pilots          : A2 4/4 + A3 4/4")
print("Production contract       : LOCKED")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Phase 05D A2 seed42       : AUTHORIZED")
print("Phase 05E A3 seed42       : AUTHORIZED")
print("Full training started     : False")
print("Proposal alignment        : ALIGNED")
print("Test Set status           : SEALED")
print_project_progress(
    cell=7,
    total=7,
    current="Phase 05C complete",
    next_step=(
        "Run Phase 05D A2 and Phase 05E A3 seed42, "
        "then Phase 05F ablation gate"
    ),
    thesis_alignment=(
        "KD improvement readiness is complete; mobile deployment "
        "remains pending until the winning checkpoint is locked."
    ),
)


PHASE 05C CELL 7 STATUS
Status                    : COMPLETE
Data pipeline             : PASSED
A2/A3 optimizer engine    : PASSED
Validation/Beam-3 engine  : PASSED
Discarded pilots          : A2 4/4 + A3 4/4
Production contract       : LOCKED
Manifest rows             : 12
Manifest verification     : ALL PASSED
Phase 05D A2 seed42       : AUTHORIZED
Phase 05E A3 seed42       : AUTHORIZED
Full training started     : False
Proposal alignment        : ALIGNED
Test Set status           : SEALED
------------------------------------------------------------------------
CURRENT PROJECT PROGRESS
------------------------------------------------------------------------
Completed                 : Phase 1 / 2 / 3 / 4 / 05A / 05B
Current                   : Phase 05C - Phase 05C complete
Notebook progress         : 7 / 7 code cells
Next                      : Run Phase 05D A2 and Phase 05E A3 seed42, then Phase 05F ablation gate
Thesis alignment          : KD improvement readiness is complete; mo